# Assignment 3 - Logan Bolton

Acknowledgement: I used Opus 4.6 to help me debug and improve my code for this assignment. 

# Word Embeddings

In [5]:
import gensim.downloader as api
wv = api.load('glove-twitter-50')
import numpy as np

## 1a

In [24]:
words = ['dog', 'bark', 'tree', 'bank', 'river', 'money']
n = len(words)

sim_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        sim_matrix[i][j] = wv.similarity(words[i], words[j])
 
header = f"{'':>8}" + "".join(f"{w:>8}" for w in words)
print(header)
for i in range(n):
    row = f"{words[i]:>8}" + "".join(f"{sim_matrix[i][j]:>8.4f}" for j in range(n))
    print(row)

             dog    bark    tree    bank   river   money
     dog  1.0000  0.5938  0.7138  0.3482  0.4012  0.5751
    bark  0.5938  1.0000  0.5459  0.0401  0.2666  0.2910
    tree  0.7138  0.5459  1.0000  0.3495  0.4871  0.5101
    bank  0.3482  0.0401  0.3495  1.0000  0.3199  0.6747
   river  0.4012  0.2666  0.4871  0.3199  1.0000  0.3378
   money  0.5751  0.2910  0.5101  0.6747  0.3378  1.0000


## 1b

In [15]:
from gensim.models import FastText
from gensim.test.utils import common_texts
ft = FastText(vector_size=50, window=5, min_count=1, sentences=common_texts, epochs=10)
 
sim_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        sim_matrix[i][j] = ft.wv.similarity(words[i], words[j])
 
header = f"{'':>8}" + "".join(f"{w:>8}" for w in words)
print(header)
for i in range(n):
    row = f"{words[i]:>8}" + "".join(f"{sim_matrix[i][j]:>8.4f}" for j in range(n))
    print(row)
 


             dog    bark    tree    bank   river   money
     dog  1.0000  0.1078 -0.1700  0.0313 -0.0135 -0.1112
    bark  0.1078  1.0000  0.2076  0.1696  0.0867 -0.0468
    tree -0.1700  0.2076  1.0000  0.0357  0.0653 -0.2631
    bank  0.0313  0.1696  0.0357  1.0000  0.2033 -0.0164
   river -0.0135  0.0867  0.0653  0.2033  1.0000 -0.1227
   money -0.1112 -0.0468 -0.2631 -0.0164 -0.1227  1.0000


## 1c

I think GloVE captured the word embeddings more accurately. FastText has very low similarity between most words, even in cases like bank and money (-0.0164) while GloVE captures it more accurately with 0.6747.

# LSTM

In [ ]:
import re
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from datasets import load_dataset
from collections import Counter

VOCAB_SIZE = 15000
MAX_LEN = 40
BATCH_SIZE = 256
NUM_CLASSES = 20
DEVICE = "cpu"

ds = load_dataset("cardiffnlp/tweet_eval", "emoji", split="train")
texts, labels = ds["text"], ds["label"]

def clean(text):
    text = re.sub(r"http\S+|www\S+", "", text)   # URLs
    text = re.sub(r"@\w+", "", text)              # mentions
    text = re.sub(r"#\w+", "", text)              # hashtags
    text = re.sub(r"[^a-zA-Z\s]", "", text)       # special characters
    return text.lower().strip()

texts = [clean(t) for t in texts]
counter = Counter(w for t in texts for w in t.split())
vocab = {w: i+2 for i, (w, _) in enumerate(counter.most_common(VOCAB_SIZE))}

def encode(text):
    ids = [vocab.get(w, 1) for w in text.split()][:MAX_LEN]
    return ids + [0] * (MAX_LEN - len(ids))

X = torch.tensor([encode(t) for t in texts])
y_onehot = F.one_hot(torch.tensor(labels), num_classes=NUM_CLASSES).float()

n = len(X)
idx = torch.randperm(n)
X, y_onehot = X[idx], y_onehot[idx]

n_train = int(0.8 * n)
n_val = int(0.1 * n)

X_train, y_train = X[:n_train], y_onehot[:n_train]
X_val, y_val = X[n_train:n_train+n_val], y_onehot[n_train:n_train+n_val]
X_test, y_test = X[n_train+n_val:], y_onehot[n_train+n_val:]

print(f"Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}")

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=BATCH_SIZE)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=BATCH_SIZE)

EMBED_DIM = 50
HIDDEN_DIM = 128
EMOJIS = ["❤️","😍","😂","💕","🔥","😊","😎","✨","💙","😘","📷","🇺🇸","☀️","💜","😉","💯","😁","🎄","📸","😜"]


Train: 36000  Val: 4500  Test: 4500


## Baseline LSTM

In [ ]:
class EmojiLSTM(nn.Module):
    def __init__(self, dropout=0.2):
        super().__init__()
        self.emb = nn.Embedding(VOCAB_SIZE + 2, EMBED_DIM, padding_idx=0)
        self.lstm = nn.LSTM(EMBED_DIM, HIDDEN_DIM, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(HIDDEN_DIM * 2, NUM_CLASSES)

    def forward(self, x):
        x = self.emb(x)
        _, (h, _) = self.lstm(x)
        h = torch.cat([h[-2], h[-1]], dim=1)
        return self.fc(self.drop(h))

def get_acc(model, loader):
    model.eval()
    correct = sum(
        (model(xb.to(DEVICE)).argmax(1) == yb.to(DEVICE).argmax(1)).sum().item()
        for xb, yb in loader
    )
    return correct

epoch_options = [3, 5, 10]
dropout_options = [0.2, 0.3, 0.5]

results = []

for drop in dropout_options:
    for max_ep in epoch_options:
        print(f"\n{'='*50}")
        print(f"Dropout={drop}  Epochs={max_ep}")
        print(f"{'='*50}")

        model = EmojiLSTM(dropout=drop).to(DEVICE)
        opt = torch.optim.Adam(model.parameters(), lr=1e-3)
        loss_fn = nn.BCEWithLogitsLoss()

        for epoch in range(1, max_ep + 1):
            model.train()
            total_loss = 0
            for xb, yb in train_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                loss = loss_fn(model(xb), yb)
                opt.zero_grad()
                loss.backward()
                opt.step()
                total_loss += loss.item()
            avg_loss = total_loss / len(train_loader)
            val_acc = get_acc(model, val_loader) / len(X_val)
            print(f"  Epoch {epoch}/{max_ep}  loss={avg_loss:.3f}  val_acc={val_acc:.3f}")

        test_acc = get_acc(model, test_loader) / len(X_test)
        print(f"  >> Test accuracy: {test_acc:.3f}")

        results.append({
            "dropout": drop,
            "epochs": max_ep,
            "train_loss": avg_loss,
            "val_acc": val_acc,
            "test_acc": test_acc,
        })

print(f"\n{'='*60}")
print(f"{'Dropout':<10}{'Epochs':<10}{'Train Loss':<12}{'Val Acc':<10}{'Test Acc':<10}")
print(f"{'='*60}")
for r in results:
    print(f"{r['dropout']:<10}{r['epochs']:<10}{r['train_loss']:<12.3f}{r['val_acc']:<10.3f}{r['test_acc']:<10.3f}")

best = max(results, key=lambda r: r["val_acc"])
print(f"\nBest config: Dropout={best['dropout']}, Epochs={best['epochs']}, Val Acc={best['val_acc']:.3f}")

print("\nSample predictions (last trained model):")
model.eval()
for i in range(10):
    x = X_test[i].unsqueeze(0).to(DEVICE)
    pred = model(x).argmax(1).item()
    true = y_test[i].argmax(0).item()
    mark = "✓" if pred == true else "✗"
    print(f"  {mark} pred: {EMOJIS[pred]}  true: {EMOJIS[true]}")

Train: 36000  Val: 4500  Test: 4500

Dropout=0.2  Epochs=3
  Epoch 1/3  loss=0.236  val_acc=0.190
  Epoch 2/3  loss=0.185  val_acc=0.197
  Epoch 3/3  loss=0.182  val_acc=0.210
  >> Test accuracy: 0.238

Dropout=0.2  Epochs=5
  Epoch 1/5  loss=0.231  val_acc=0.190
  Epoch 2/5  loss=0.184  val_acc=0.197
  Epoch 3/5  loss=0.180  val_acc=0.215
  Epoch 4/5  loss=0.177  val_acc=0.222
  Epoch 5/5  loss=0.174  val_acc=0.209
  >> Test accuracy: 0.236

Dropout=0.2  Epochs=10
  Epoch 1/10  loss=0.236  val_acc=0.190
  Epoch 2/10  loss=0.185  val_acc=0.195
  Epoch 3/10  loss=0.182  val_acc=0.210
  Epoch 4/10  loss=0.179  val_acc=0.218
  Epoch 5/10  loss=0.176  val_acc=0.228
  Epoch 6/10  loss=0.173  val_acc=0.234
  Epoch 7/10  loss=0.171  val_acc=0.240
  Epoch 8/10  loss=0.168  val_acc=0.244
  Epoch 9/10  loss=0.166  val_acc=0.246
  Epoch 10/10  loss=0.163  val_acc=0.251
  >> Test accuracy: 0.265

Dropout=0.3  Epochs=3
  Epoch 1/3  loss=0.233  val_acc=0.190
  Epoch 2/3  loss=0.185  val_acc=0.190
  

## FastText LSTM

In [ ]:
import os
import numpy as np
import urllib.request
import zipfile

FASTTEXT_DIM = 50
FASTTEXT_FILE = "wiki-news-300d-1M-subword.vec"
FASTTEXT_ZIP = "fasttext.zip"
FASTTEXT_URL = "https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M-subword.vec.zip"

if not os.path.exists(FASTTEXT_FILE):
    print("Downloading FastText vectors (~600MB, one-time)...")
    urllib.request.urlretrieve(FASTTEXT_URL, FASTTEXT_ZIP)
    with zipfile.ZipFile(FASTTEXT_ZIP) as z:
        z.extractall(".")
    print("Done.")

print("Loading FastText embeddings into vocab matrix...")
embedding_matrix = np.zeros((VOCAB_SIZE + 2, FASTTEXT_DIM))
found = 0
with open(FASTTEXT_FILE, "r", encoding="utf-8") as f:
    next(f)  # skip header
    for line in f:
        parts = line.rstrip().split(" ")
        word = parts[0]
        if word in vocab:
            vec = np.array(parts[1:FASTTEXT_DIM+1], dtype=np.float32)
            if len(vec) == FASTTEXT_DIM:
                embedding_matrix[vocab[word]] = vec
                found += 1
print(f"Matched {found}/{len(vocab)} vocab words")

class EmojiLSTM_FastText(nn.Module):
    def __init__(self, embedding_matrix, dropout=0.2):
        super().__init__()
        n_emb, emb_dim = embedding_matrix.shape
        self.emb = nn.Embedding(n_emb, emb_dim, padding_idx=0)
        self.emb.weight = nn.Parameter(torch.tensor(embedding_matrix, dtype=torch.float32))
        self.lstm = nn.LSTM(emb_dim, HIDDEN_DIM, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(HIDDEN_DIM * 2, NUM_CLASSES)

    def forward(self, x):
        x = self.emb(x)
        _, (h, _) = self.lstm(x)
        h = torch.cat([h[-2], h[-1]], dim=1)
        return self.fc(self.drop(h))

def get_acc(model, loader):
    model.eval()
    correct = sum(
        (model(xb.to(DEVICE)).argmax(1) == yb.to(DEVICE).argmax(1)).sum().item()
        for xb, yb in loader
    )
    return correct

epoch_options = [3, 5, 10]
dropout_options = [0.2, 0.3, 0.5]
ft_results = []

for drop in dropout_options:
    for max_ep in epoch_options:
        print(f"\n{'='*50}")
        print(f"[FastText] Dropout={drop}  Epochs={max_ep}")
        print(f"{'='*50}")

        model = EmojiLSTM_FastText(embedding_matrix, dropout=drop).to(DEVICE)
        opt = torch.optim.Adam(model.parameters(), lr=1e-3)
        loss_fn = nn.BCEWithLogitsLoss()

        for epoch in range(1, max_ep + 1):
            model.train()
            total_loss = 0
            for xb, yb in train_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                loss = loss_fn(model(xb), yb)
                opt.zero_grad()
                loss.backward()
                opt.step()
                total_loss += loss.item()
            avg_loss = total_loss / len(train_loader)
            val_acc = get_acc(model, val_loader) / len(X_val)
            print(f"  Epoch {epoch}/{max_ep}  loss={avg_loss:.3f}  val_acc={val_acc:.3f}")

        test_acc = get_acc(model, test_loader) / len(X_test)
        print(f"  >> Test accuracy: {test_acc:.3f}")

        ft_results.append({
            "dropout": drop,
            "epochs": max_ep,
            "train_loss": avg_loss,
            "val_acc": val_acc,
            "test_acc": test_acc,
        })

print(f"\n{'='*60}")
print(f"{'Dropout':<10}{'Epochs':<10}{'Train Loss':<12}{'Val Acc':<10}{'Test Acc':<10}")
print(f"{'='*60}")
for r in ft_results:
    print(f"{r['dropout']:<10}{r['epochs']:<10}{r['train_loss']:<12.3f}{r['val_acc']:<10.3f}{r['test_acc']:<10.3f}")

best_ft = max(ft_results, key=lambda r: r["val_acc"])
print(f"\nBest FastText config: Dropout={best_ft['dropout']}, Epochs={best_ft['epochs']}, Val={best_ft['val_acc']:.3f}, Test={best_ft['test_acc']:.3f}")

Done.
Loading FastText embeddings into vocab matrix...
Matched 11959/15000 vocab words

[FastText] Dropout=0.2  Epochs=3
  Epoch 1/3  loss=0.232  val_acc=0.211
  Epoch 2/3  loss=0.186  val_acc=0.211
  Epoch 3/3  loss=0.184  val_acc=0.234
  >> Test accuracy: 0.227

[FastText] Dropout=0.2  Epochs=5
  Epoch 1/5  loss=0.235  val_acc=0.211
  Epoch 2/5  loss=0.185  val_acc=0.211
  Epoch 3/5  loss=0.183  val_acc=0.233
  Epoch 4/5  loss=0.175  val_acc=0.258
  Epoch 5/5  loss=0.169  val_acc=0.256
  >> Test accuracy: 0.255

[FastText] Dropout=0.2  Epochs=10
  Epoch 1/10  loss=0.233  val_acc=0.211
  Epoch 2/10  loss=0.186  val_acc=0.211
  Epoch 3/10  loss=0.183  val_acc=0.234
  Epoch 4/10  loss=0.176  val_acc=0.256
  Epoch 5/10  loss=0.169  val_acc=0.262
  Epoch 6/10  loss=0.163  val_acc=0.257
  Epoch 7/10  loss=0.159  val_acc=0.258
  Epoch 8/10  loss=0.155  val_acc=0.254
  Epoch 9/10  loss=0.151  val_acc=0.261
  Epoch 10/10  loss=0.147  val_acc=0.250
  >> Test accuracy: 0.250

[FastText] Dropout

## Hyperparameter Analysis

### Baseline Model

| Dropout | Epochs | Train Loss | Val Acc | Test Acc | Notes |
|---------|--------|------------|---------|----------|-------|
| 0.2 | 3 | 0.182 | 0.210 | 0.238 | Undertrained |
| 0.2 | 5 | 0.174 | 0.209 | 0.236 |slight instability |
| 0.2 | 10 | 0.163 | 0.251 | 0.265 | Steady improvement |
| 0.3 | 3 | 0.183 | 0.204 | 0.230 | barely learning |
| 0.3 | 5 | 0.175 | 0.229 | 0.250 | Improving steadily |
| 0.3 | 10 | 0.164 | 0.252 | 0.266 | Best overall, still improving at epoch 10 |
| 0.5 | 3 | 0.184 | 0.208 | 0.231 | slows early learning |
| 0.5 | 5 | 0.178 | 0.226 | 0.251 | Converging slower|
| 0.5 | 10 | 0.167 | 0.249 | 0.263 | Slightly worse than 0.3 |

### FastText Embeddings

### FastText Model

| Dropout | Epochs | Train Loss | Val Acc | Test Acc | Notes |
|---------|--------|------------|---------|----------|-------|
| 0.2 | 3 | 0.184 | 0.234 | 0.227 | Undertrained |
| 0.2 | 5 | 0.169 | 0.256 | 0.255 | Decent |
| 0.2 | 10 | 0.147 | 0.250 | 0.250 | Overfitting|
| 0.3 | 3 | 0.185 | 0.226 | 0.221 | Slow start |
| 0.3 | 5 | 0.172 | 0.259 | 0.254 |  converging well |
| 0.3 | 10 | 0.145 | 0.263 | 0.252 | Val peaked at epoch 8 then dropped, overfitting |
| 0.5 | 3 | 0.186 | 0.210 | 0.208 | Worst result,  |
| **0.5** | **5** | **0.172** | **0.268** | **0.261** | **Best config, sweet spot of regularization** |
| 0.5 | 10 | 0.153 | 0.255 | 0.255 | Overfitting |


### Analysis

a) The baseline LSTM slightly outperformed FastText on test accuracy (0.266 vs 0.261). FastText converges faster thanks to pre-trained word knowledge, but is more prone to overfitting, especially with longer training.

b) More epochs helped the baseline steadily improve, while FastText tended to peak early then decline. Dropout had a mild effect on the baseline, but mattered more for FastText, where higher dropout helped prevent overfitting. The best baseline config was dropout=0.3 with 10 epochs, while FastText worked best with dropout=0.5 and just 5 epochs.

c) The model over-predicted the heart emoji, likely because it is the most common class. Many errors involved similar emojis like ❤️ vs 💕 or 😂 vs 😜, showing the model struggles to distinguish similar classes.